# Weather and Exchange Rate Preprocessing

Cleans daily weather and USD/LKR exchange rate series.

In [ ]:
import numpy as np
import pandas as pd
from paths import PREPROCESSED_PATHS, RAW_PATHS, save_dataset


In [ ]:
fx = pd.read_csv(RAW_PATHS['exchange_rates'])
if 'LKR=X' in fx.columns:
    fx = fx.drop(columns=['LKR=X'])
fx = fx.rename(columns={'date': 'Report_Date', 'USD_LKR_Rate': 'Exchange_Rate'})
fx['Report_Date'] = pd.to_datetime(fx['Report_Date'])
fx = fx.set_index('Report_Date').sort_index()
fx['Exchange_Rate'] = pd.to_numeric(fx['Exchange_Rate'], errors='coerce').ffill()
save_dataset(fx, PREPROCESSED_PATHS['exchange_daily'])
print('Exchange shape:', fx.shape)


In [ ]:
weather = pd.read_csv(RAW_PATHS['weather'])
weather['Report_Date'] = pd.to_datetime(weather['Report_Date'])
weather = weather.set_index('Report_Date').sort_index()
for col in ['Rainfall_mm', 'Temp_Max_C', 'Temp_Min_C']:
    if col in weather.columns:
        weather[col] = pd.to_numeric(weather[col], errors='coerce')
if 'Rainfall_mm' in weather.columns:
    weather['Rainfall_mm'] = weather['Rainfall_mm'].fillna(0)
    weather['Rainfall_14d_Cumulative'] = weather['Rainfall_mm'].rolling(window=14, min_periods=1).sum()
for col in ['Temp_Max_C', 'Temp_Min_C']:
    if col in weather.columns:
        weather[col] = weather[col].interpolate(method='linear')
if 'Temp_Max_C' in weather.columns:
    weather['Is_Extreme_Heat'] = np.where(weather['Temp_Max_C'] > 35.0, 1, 0)
save_dataset(weather, PREPROCESSED_PATHS['weather_daily'])
print('Weather shape:', weather.shape)
